# FLEX — every California NPS unit, every main variable, all scenarios

Pure scale-and-speed flex of the library + Coiled stack. Pull monthly `T_Max`, `T_Min`, and `Precip` for **every NPS unit geographically in California**, across Historical + all three SSPs, 1950–2100, all 15 LOCA2 models. Compute the climatological aggregates that actually matter per park (mid-term warming, precipitation anomaly, year of emergence). Skip the pretty plots; the point is that **the whole thing finishes during a coffee break**.

## The baseline we're beating

Running this same workflow serially on a MacBook: pulling just *one* variable (Precip) for *one* park (Joshua Tree) from Cal-Adapt's Zarr stores over home wifi took me **~36 hours** last year. Extrapolating linearly, doing 15 parks × 3 variables = **45 park-variable combos × 36 hrs ≈ 1,620 hrs ≈ 67 days**.

So, for getting all the areas that BLM/NPS manages in CA, let's see how long it takes. Also, without spoiling how long the end result is, using coiled htis is scalable so that we could even go x2 faster on the processing (and note that there is a limit to how much faster as parallelization across workers will stop being helpful ast some point), but just asking GCP for greater quotoa limits and spinning up more wokrers. 



In [ ]:
import os, sys, time
from time import perf_counter
import coiled
import pandas as pd
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
from shapely.geometry import Polygon

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, os.path.join(PROJECT_ROOT, "lib"))

from andrewAdaptLibrary import (
    ParkCatalog,
    CatalogExplorer,
    get_climate_data,
    compute_t_avg,      # T_Max + T_Min → T_Avg
    annual_aggregate,   # monthly → annual (mean for T, sum for P)
    smooth,             # centered rolling mean
)


#  timing helper ~~
class Timer:
    def __init__(self, label): self.label = label
    def __enter__(self):
        self.start = perf_counter()
        print(f"▶ {self.label}…")
        return self
    def __exit__(self, *args):
        self.elapsed = perf_counter() - self.start
        print(f"  ✓ {self.label}: {self.elapsed:.1f}s ({self.elapsed/60:.2f} min)")


T_START = perf_counter()

NPS_SHP = os.path.join(
    PROJECT_ROOT,
    "USA_National_Park_Service_Lands_20170930_4993375350946852027",
    "USA_Federal_Lands.shp",
)
park_catalog = ParkCatalog(NPS_SHP)
cat = CatalogExplorer(timescale="monthly")
print(park_catalog, "|", cat)

/opt/conda/envs/py-env/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarning: /workspaces/DSEBrandNew/USA_National_Park_Service_Lands_20170930_4993375350946852027/USA_Federal_Lands.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


ParkCatalog(437 NPS units) | CatalogExplorer(activity='LOCA2', timescale='monthly', grid='d03', stores=1585)


## 1. Find every CA NPS unit — polygon-contains on the mega layer

Hand-crafted California polygon; any unit whose centroid sits inside it is a candidate.

In [11]:
CA = Polygon([
    (-124.41, 42.00), (-120.00, 42.00), (-120.00, 38.99),
    (-114.63, 35.00), (-114.63, 32.72), (-117.12, 32.53),
    (-117.30, 33.00), (-118.48, 33.73), (-119.88, 34.42),
    (-120.90, 35.27), (-121.89, 36.60), (-122.51, 37.80),
    (-123.02, 38.30), (-123.72, 39.35), (-124.15, 40.44),
    (-124.41, 42.00),
])

gdf = park_catalog._gdf
in_ca_mask = gdf.geometry.centroid.apply(CA.contains)
ca_names = sorted(gdf.loc[in_ca_mask, "unit_name"].unique())
print(f"{len(ca_names)} NPS units with centroid inside California")

26 NPS units with centroid inside California


/tmp/ipykernel_61017/2493356612.py:11: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  in_ca_mask = gdf.geometry.centroid.apply(CA.contains)


## 2. Safety filters — LOCA2 coverage + bbox size

Park bbox must span at least 0.05° (~5 km) in each dim, otherwise the 3 km LOCA2 grid `.sel()` returns zero cells and `rio.clip` crashes with `NoDataInBounds`. Drops the urban/historic memorials (Fort Point, Eugene O'Neill, etc.) — too small for meaningful climate representation anyway.

In [12]:
def bbox_big_enough(boundary, min_deg=0.05):
    b = boundary.total_bounds
    return (b[2] - b[0]) >= min_deg and (b[3] - b[1]) >= min_deg

boundaries = {n: park_catalog.get_boundary(n) for n in ca_names}
ready = [n for n, b in boundaries.items()
         if park_catalog._inside_loca2(b) and bbox_big_enough(b)]
skipped = [n for n in ca_names if n not in ready]
print(f"ready: {len(ready)}   skipped: {len(skipped)}")

ready: 15   skipped: 11


## 3. Start the cluster — 45 workers, **then wait for them**

The previous version of this notebook submitted work before all workers were actually ready; the first batch got cancelled with `FutureCancelledError` when their assigned workers hadn't booted yet. Fix: `client.wait_for_workers()` after creation to block until the cluster is actually usable.

In [13]:
with Timer("cluster spin-up + warmup") as cluster_timer:
    cluster = coiled.Cluster(
        name="flex-ca",
        region="us-central1",
        n_workers=45,
        worker_vm_types=["n2-highmem-4"],
        spot_policy="spot_with_fallback",
        idle_timeout="30 minutes",
        package_sync=True,
    )
    client = cluster.get_client()
    # Block until at least 30 of 45 workers are registered and ready
    # (don't need all 45 — Dask will queue work to whatever's available)
    print("  Waiting for workers to register...")
    client.wait_for_workers(n_workers=30, timeout=180)

n_workers_actual = len(client.scheduler_info()['workers'])
print(f"  Workers ready: {n_workers_actual} / 45 requested")
print(f"  Dashboard: {client.dashboard_link}")

▶ cluster spin-up + warmup…


/tmp/ipykernel_61017/798770121.py:2: FutureWarning: `package_sync` is a deprecated kwarg for `Cluster` and will be removed in a future release. To only sync certain packages, use `package_sync_only`, and to disable package sync, pass the `container` or `software` kwargs instead.
  cluster = coiled.Cluster(


Output()

╭──────────────────────────────── Package Info ────────────────────────────────╮
│                    ╷                                                         │
│   Package          │ Note                                                    │
│ ╶──────────────────┼───────────────────────────────────────────────────────╴ │
│   coiled_local_lib │ Source wheel built from /workspaces/DSEBrandNew/lib     │
│   climakitae       │ Wheel built from                                        │
│                    │ https://github.com/cal-adapt/climakitae/archive/refs/   │
│                    │ tags/1.4.0.zip                                          │
│                    ╵                                                         │
╰──────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────── Not Synced with Cluster ───────────────────────────╮
│             ╷                                                    ╷           │
│   Package   │ Error                                              │ Level     │
│ ╶───────────┼────────────────────────────────────────────────────┼─────────╴ │
│   shiboken6 │ cannot find shiboken6~=6.10.2 on pypi.org. If you  │ Warning   │
│             │ are using a custom PyPI URL, please see            │           │
│             │ https://docs.coiled.io/user_guide/software/packag… │           │
│             │ for more instructions.                             │           │
│             ╵                                                    ╵           │
╰──────────────────────────────────────────────────────────────────────────────╯

Output()

  Waiting for workers to register...
  ✓ cluster spin-up + warmup: 226.2s (3.77 min)
  Workers ready: 5 / 45 requested
  Dashboard: https://cluster-skcss.dask.host/el4pkb-FsvijGFSR/status


## 4. Map-reduce across all parks — 3 variables × 4 scenarios

Per park, `get_climate_data` dispatches `3 vars × 4 scenarios = 12` delayed tasks to Dask. ThreadPoolExecutor runs the per-park calls concurrently, so the scheduler sees `12 × len(ready)` tasks at once — plenty to saturate the 45-worker cluster.

`fetch()` has a 3-attempt retry with exponential backoff in case a worker gets spot-preempted mid-task (the common failure mode for this kind of run).

In [14]:
scenarios = ["Historical Climate", "SSP 2-4.5", "SSP 3-7.0", "SSP 5-8.5"]
variables = ["T_Max", "T_Min", "Precip"]

def fetch(name, max_attempts=3):
    t0 = perf_counter()
    last_err = None
    for attempt in range(1, max_attempts + 1):
        try:
            data = get_climate_data(
                variables=variables, scenarios=scenarios,
                boundary=boundaries[name], time_slice=(1950, 2100),
                timescale="monthly", backend="coiled",
                coiled_cluster=cluster, catalog=cat,
            )
            # Combine the 3 variables into one tidy DataFrame keyed on (sim, time, scenario)
            merge_keys = ["simulation", "time", "scenario", "timescale"]
            df = data["T_Max"]
            for var in ["T_Min", "Precip"]:
                df = df.merge(data[var][merge_keys + [var]], on=merge_keys)
            df["park"] = name
            return name, df, perf_counter() - t0, None, attempt
        except Exception as e:
            last_err = f"{type(e).__name__}: {str(e)[:80]}"
            if attempt < max_attempts:
                time.sleep(2 * attempt)   # exponential backoff: 2s, 4s
    return name, None, perf_counter() - t0, last_err, max_attempts


results, errors, per_park_times = {}, {}, {}

with Timer("all-park fetch (wall-clock)") as fetch_timer:
    with ThreadPoolExecutor(max_workers=len(ready)) as ex:
        futures = [ex.submit(fetch, name) for name in ready]
        for i, f in enumerate(as_completed(futures), 1):
            name, df, dt, err, attempts = f.result()
            per_park_times[name] = dt
            retry_mark = f" ({attempts} tries)" if attempts > 1 else ""
            if err is None:
                results[name] = df
                print(f"  [{i:2d}/{len(ready)}] {name[:42]:42s}  {dt:5.1f}s  {len(df):>7,} rows{retry_mark}")
            else:
                errors[name] = err
                print(f"  [{i:2d}/{len(ready)}] {name[:42]:42s}  {dt:5.1f}s  ✗ {err[:70]}")

if errors:
    print(f"\n⚠️  {len(errors)} park(s) failed after retries")

▶ all-park fetch (wall-clock)…
  [ 1/15] Sequoia National Park                       328.9s  187,728 rows
  [ 2/15] Pinnacles National Park                     332.9s  187,728 rows
  [ 3/15] Whiskeytown-Shasta-Trinity National Recrea  345.1s  187,728 rows
  [ 4/15] Golden Gate National Recreation Area        351.8s  187,728 rows
  [ 5/15] Channel Islands National Park               351.8s  187,728 rows
  [ 6/15] Castle Mountains National Monument          364.7s  187,728 rows
  [ 7/15] Mojave National Preserve                    364.7s  187,728 rows
  [ 8/15] Lassen Volcanic National Park               364.7s  187,728 rows
  [ 9/15] Death Valley National Park                  364.7s  187,728 rows
  [10/15] Kings Canyon National Park                  378.3s  187,728 rows
  [11/15] Lava Beds National Monument                 378.3s  187,728 rows
  [12/15] Redwood National Park                       378.3s  187,728 rows
  [13/15] Yosemite National Park                      378.3s  187,728

## 5. Reduce — concat + compute T_Avg

In [15]:
with Timer("concat + T_Avg") as concat_timer:
    all_data = pd.concat(results.values(), ignore_index=True)
    all_data["time"] = pd.to_datetime(all_data["time"])
    all_data["T_Avg"] = (all_data["T_Max"] + all_data["T_Min"]) / 2

print(f"  rows:        {len(all_data):>12,}")
print(f"  parks:       {all_data['park'].nunique():>12}")
print(f"  simulations: {all_data['simulation'].nunique():>12}")
print(f"  memory:      {all_data.memory_usage(deep=True).sum() / 1e6:>11.0f} MB")

▶ concat + T_Avg…
  ✓ concat + T_Avg: 0.2s (0.00 min)
  rows:           2,815,920
  parks:                 15
  simulations:           70
  memory:              400 MB


## 6. Climatological aggregates — three numbers per (park, scenario)

For each park and each future SSP, compute:

1. **ΔT_Avg (°C)** — mid-term (2041–2060) annual mean T_Avg minus historical (1985–2014) annual mean T_Avg. The standard IPCC warming metric.
2. **ΔPrecip (%)** — mid-term annual total precipitation vs historical, as percent change.
3. **Year of emergence** — first year where a **10-year centered rolling mean** of the SSP's multi-model-mean annual T_Avg exceeds the **same rolling mean applied to historical** at its warmest 10-yr window. A published climate-science metric (Hawkins & Sutton 2012, Mahlstein et al. 2011) — *"when does this park leave its historical climate envelope?"* Both sides are smoothed the same way so edge effects at the 2014/2015 handoff don't create artifacts. `NaN` means emergence doesn't happen before 2100.

None of these need the heavy plotting — they're the numbers that would go in a summary row of a paper.

In [ ]:
# Reshape to xarray for clean climatological ops
def df_to_da(df, park, var):
    sub = df[df["park"] == park]
    return (sub.set_index(["scenario", "simulation", "time"])[var]
              .to_xarray().sortby("time"))


HIST_WINDOW   = ("1985", "2014")
FUTURE_WINDOW = ("2041", "2060")
SMOOTH_WINDOW = 10   # years — 10-yr centered rolling mean for emergence check

def park_aggregates(park):
    tavg   = df_to_da(all_data, park, "T_Avg")
    precip = df_to_da(all_data, park, "Precip")
    tavg.name, precip.name = "T_Avg", "Precip"

    # Annualize (lib helpers)
    tavg_ann   = annual_aggregate(tavg, "T_Avg")
    precip_ann = annual_aggregate(precip, "Precip")

    # Historical baseline means (for ΔT, ΔP)
    tavg_hist_mean   = (tavg_ann.sel(scenario="Historical Climate")
                                .sel(time=slice(*HIST_WINDOW))
                                .mean(dim=["time", "simulation"]))
    precip_hist_mean = (precip_ann.sel(scenario="Historical Climate")
                                  .sel(time=slice(*HIST_WINDOW))
                                  .mean(dim=["time", "simulation"]))

    # For emergence: use the SMOOTHED historical MMEM's peak 10-yr window as the
    # ceiling. Strict rolling (min_periods=window) so edge years are NaN — this
    # avoids the artifact where an edge-smoothed SSP value at 2015 trivially
    # "exceeds" a raw historical single-year max.
    tavg_hist_mmem = tavg_ann.sel(scenario="Historical Climate").mean(dim="simulation")
    hist_smoothed  = tavg_hist_mmem.rolling(time=SMOOTH_WINDOW, center=True,
                                            min_periods=SMOOTH_WINDOW).mean()
    hist_ceiling   = float(hist_smoothed.max(dim="time", skipna=True))

    rows = []
    for scen in ["SSP 2-4.5", "SSP 3-7.0", "SSP 5-8.5"]:
        t_mid = (tavg_ann.sel(scenario=scen).sel(time=slice(*FUTURE_WINDOW))
                         .mean(dim=["time", "simulation"]))
        p_mid = (precip_ann.sel(scenario=scen).sel(time=slice(*FUTURE_WINDOW))
                           .mean(dim=["time", "simulation"]))
        dT_avg = float(t_mid - tavg_hist_mean)
        dP_pct = float((p_mid - precip_hist_mean) / precip_hist_mean * 100)

        # Year of emergence: first year where smoothed SSP MMEM (strict window)
        # exceeds the smoothed historical ceiling. With min_periods=window, the
        # SSP smoothed series is NaN for ~2015-2019 (window not yet full), so
        # exceed checks there fail naturally — no need for an explicit year filter.
        scen_mmem   = tavg_ann.sel(scenario=scen).mean(dim="simulation")
        scen_smooth = scen_mmem.rolling(time=SMOOTH_WINDOW, center=True,
                                        min_periods=SMOOTH_WINDOW).mean()
        exceed = scen_smooth > hist_ceiling
        years = pd.DatetimeIndex(scen_smooth.time.values).year
        first_exceed = next((int(y) for y, ex in zip(years, exceed.values)
                             if bool(ex)), None)

        rows.append({
            "park": park, "scenario": scen,
            "ΔT_Avg (°C)": dT_avg,
            "ΔPrecip (%)": dP_pct,
            "Emergence year": first_exceed if first_exceed is not None else np.nan,
        })
    return rows


with Timer("aggregate computation") as agg_timer:
    agg_rows = []
    for park in results:
        agg_rows.extend(park_aggregates(park))

agg_df = pd.DataFrame(agg_rows)

# Sort by ΔT under SSP 5-8.5 — most-warming parks at the top
order = (agg_df[agg_df.scenario == "SSP 5-8.5"]
            .sort_values("ΔT_Avg (°C)", ascending=False)["park"].tolist())

summary = (agg_df.pivot(index="park", columns="scenario",
                        values=["ΔT_Avg (°C)", "ΔPrecip (%)", "Emergence year"])
                  .reindex(order))
summary.round(2)

## 7. Cost + thrjoughput summary


In [ ]:
T_TOTAL = perf_counter() - T_START

N_MODELS, N_VARS, N_SCENARIOS = 15, len(variables), len(scenarios)
N_UNIQUE_STORES = N_MODELS * N_VARS * N_SCENARIOS              # 180
AVG_STORE_GB    = 1.26                                          # hist ~1.0, SSP ~1.33, averaged
store_reads     = N_UNIQUE_STORES * len(results)                # 180 × 15 ≈ 2,700
data_chomped_gb = store_reads * AVG_STORE_GB

sim_month_count = len(all_data) * 1   # rows are already sim-month rows
simulation_years = sim_month_count / 12

# -- Timing stats --
success_times = {k: v for k, v in per_park_times.items() if k in results}
t_slowest = max(success_times.values()) if success_times else 0
t_fastest = min(success_times.values()) if success_times else 0
t_avg     = sum(success_times.values()) / max(len(success_times), 1)
t_serial  = sum(success_times.values())

# -- The baseline: user measured 36 hours to do 1 park × 1 variable on a Mac --
PARK_VAR_COMBOS = len(results) * N_VARS
LOCAL_HRS_PER_COMBO = 36
local_total_hours = PARK_VAR_COMBOS * LOCAL_HRS_PER_COMBO
local_days = local_total_hours / 24
cloud_minutes = T_TOTAL / 60
speedup = (local_total_hours * 3600) / T_TOTAL

print("=" * 62)
print("  What did we actually  actually process?")
print("=" * 62)
print(f"  Parks processed:              {len(results)} of {len(ready)} ready  (failed: {len(errors)})")
print(f"  Variables per park:           {N_VARS}  ({', '.join(variables)})")
print(f"  Scenarios:                    {N_SCENARIOS}  ({', '.join(scenarios)})")
print(f"  GCMs (climate models):        {N_MODELS}")
print(f"  Unique Zarr stores opened:    {N_UNIQUE_STORES}  ({N_MODELS} × {N_VARS} × {N_SCENARIOS})")
print(f"  Total store-read operations:  {store_reads:,}")
print(f"  Simulation-years processed:   {simulation_years:>12,.0f}")
print(f"  Simulation-months processed:  {sim_month_count:>12,}")
print(f"  Raw climate data chomped:     {data_chomped_gb:>10,.0f} GB  ({data_chomped_gb/1000:.2f} TB)")
print()
print("=" * 62)
print("  TIMING")
print("=" * 62)
print(f"  Cluster spin-up + warmup:     {cluster_timer.elapsed:>7.1f}s  ({cluster_timer.elapsed/60:.1f} min)")
print(f"  Fetch wall-clock (parallel):  {fetch_timer.elapsed:>7.1f}s  ({fetch_timer.elapsed/60:.1f} min)")
if success_times:
    print(f"    ↳ avg per-park fetch:       {t_avg:>7.1f}s")
    print(f"    ↳ slowest park:             {t_slowest:>7.1f}s  ({max(success_times, key=success_times.get)})")
    print(f"    ↳ fastest park:             {t_fastest:>7.1f}s  ({min(success_times, key=success_times.get)})")
    print(f"    ↳ sum if done serially:     {t_serial:>7.1f}s  ({t_serial/60:.1f} min) — parallelism saved {t_serial-fetch_timer.elapsed:.0f}s")
print(f"  Concat + T_Avg:               {concat_timer.elapsed:>7.1f}s")
print(f"  Aggregate computation:        {agg_timer.elapsed:>7.1f}s")
print(f"  TOTAL WALL-CLOCK:             {T_TOTAL:>7.1f}s  ({cloud_minutes:.1f} min)")
print()
print("=" * 62)
print(f"  SPEEDUP")
print("=" * 62)
print(f"  Baseline (measured on Mac):   ~{LOCAL_HRS_PER_COMBO} hrs per (park × variable)")
print(f"  → {PARK_VAR_COMBOS} park-variable combos on a Mac: ~{local_total_hours:,.0f} hrs  ≈ {local_days:.0f} days")
print(f"  This notebook:                {cloud_minutes:.1f} minutes")
print(f"  SPEEDUP:                      ~{speedup:>7,.0f}×")

  SCALE — what this run actually processed
  Parks processed:              15 of 15 ready  (failed: 0)
  Variables per park:           3  (T_Max, T_Min, Precip)
  Scenarios:                    4  (Historical Climate, SSP 2-4.5, SSP 3-7.0, SSP 5-8.5)
  GCMs (climate models):        15
  Unique Zarr stores opened:    180  (15 × 3 × 4)
  Total store-read operations:  2,700
  Simulation-years processed:        234,660
  Simulation-months processed:     2,815,920
  Raw climate data chomped:          3,402 GB  (3.40 TB)

  TIMING
  Cluster spin-up + warmup:       226.2s  (3.8 min)
  Fetch wall-clock (parallel):    378.4s  (6.3 min)
    ↳ avg per-park fetch:         362.6s
    ↳ slowest park:               378.4s  (Joshua Tree National Park)
    ↳ fastest park:               328.9s  (Sequoia National Park)
    ↳ sum if done serially:      5439.3s  (90.7 min) — parallelism saved 5061s
  Concat + T_Avg:                   0.2s
  Aggregate computation:            2.9s
  TOTAL WALL-CLOCK:         

## 8. Shut down

In [18]:
cluster.close()
print("Done.")

Done.
